In [0]:
%run
../utilities/file_utilities


In [0]:
dbutils.widgets.text("table_name", "")


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,max,concat_ws,coalesce,lit,sha2
import json
import os
spark = SparkSession.builder.appName("fep_ben_mdrnz_apg_2_databricks").getOrCreate()



try:
    # Load Postgres connection parameters from JSON file
    with open('../parameter_config/unload_apg_metdata.json', 'r') as f:
        conn_params = json.load(f)
except json.JSONDecodeError as e:
        print(f"Error parsing parameter config JSON: {e}")
        raise
except FileNotFoundError:
        print(" parameter config File not found! Check your path.")
        raise
except Exception as e:
        print(f"Unexpected error: {e}")
        raise


table_name=dbutils.widgets.get("table_name")
conn_config=conn_params['conn_config']
target_catalog=conn_params['databricks_catalog_name']
full_table_nm=f"{target_catalog}.silver.{table_name}"
checkpoint_file_path="/Volumes/benefit_modernization_dev/silver/delta_checkpoint_files"
#delta_checkpoint_file=f"abfss://benefit-modernization@enterprzmdrnzetl2025.dfs.core.windows.net/delta_checkpoint_files/{table_name}.checkpoint"
delta_checkpoint_file=f"{checkpoint_file_path}/{table_name}.checkpoint"




if(path_exists(delta_checkpoint_file)==False and not spark.catalog.tableExists(full_table_nm)):
         df=spark.read.table(f"{target_catalog}.bronze.{table_name}")
         df_hash_val=df.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256))
         df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)
         df.select(max(col("CREATED_AT")).alias("max_created_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)

elif(path_exists(delta_checkpoint_file)==False and spark.catalog.tableExists(full_table_nm)):
        df_count=spark.read.table(full_table_nm).count()
        if(df_count==0):
                df=spark.read.table(f"{target_catalog}.bronze.{table_name}")
                df_hash_val=df.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256))
                df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)
                df.select(max(col("CREATED_AT")).alias("max_created_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)
        else:
                df=spark.read.table(full_table_nm)
                
                df.select(max(col("CREATED_AT")).alias("max_created_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)

                df_cdc_silver=spark.read.table(full_table_nm)
                checkpoint_val=spark.read.csv(delta_checkpoint_file).take(1)[0][0];
                
                df_cdc_bronze=spark.read.table(f"{target_catalog}.bronze.{table_name}")
                df_cdc_bronze_final=df_cdc_bronze.filter(col("CREATED_AT")>checkpoint_val).withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df_cdc_bronze.columns]),256))
     
                df_cdc_bronze_final.createOrReplaceTempView("df_cdc_bronze")
                df_cdc_silver_final.createOrReplaceTempView("df_cdc_silver")
                df_cdc=spark.sql('''MERGE INTO df_cdc_silver as tgt
                                    USING df_cdc_bronze as src
                                    ON tgt.admissions_id=src.bronze_admissions_id
                                    WHEN MATCHED AND tgt.hash_val!=src.hash_val THEN UPDATE SET *
                                    WHEN NOT MATCHED  THEN INSERT * ''')

else:

        print("checkpoint file exists")
        df_count=spark.read.table(full_table_nm).count()
        if(df_count==0):
                df=spark.read.table(f"{target_catalog}.bronze.{table_name}")
                df_hash_val=df.withColumn("hash_val",sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256))
                df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)
                df.select(max(col("CREATED_AT")).alias("max_created_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)
        else:
                checkpoint_val=spark.read.csv(delta_checkpoint_file).take(1)[0][0];

                df_cdc_silver=spark.read.table(full_table_nm)

                df_cdc_bronze=spark.read.table(f"{target_catalog}.bronze.{table_name}").filter(col("CREATED_AT")>checkpoint_val ).filter(col("UPDATED_AT")>checkpoint_val)

  #              df_cdc_bronze_intmd=df_cdc_bronze.select(*[col(c).alias(f"bronze_{c}") for c in df_cdc_bronze.columns])

                df_cdc_bronze_final=df_cdc_bronze.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df_cdc_bronze.columns]),256))
     
                df_cdc_bronze_final.createOrReplaceTempView("df_cdc_bronze")
                df_cdc_silver.createOrReplaceTempView("df_cdc_silver")

                df_cdc=spark.sql('''MERGE INTO df_cdc_silver as tgt
                                    USING df_cdc_bronze as src
                                    ON tgt.admission_id=src.admission_id
                                    WHEN MATCHED AND tgt.hash_val!=src.hash_val THEN UPDATE SET *
                                    WHEN NOT MATCHED  THEN INSERT * ''')
      


         
      

In [0]:
a=spark.read.csv(delta_checkpoint_file).take(1)[0][0];
print(a)